# Zarr & OME-Zarr support in 🤗datasets — demo notebook

This notebook walks through everything the Zarr-support PR adds to `datasets`:

1. **`Zarr` feature** — a lazy proxy column for Zarr stores (plain arrays, groups, and OME-Zarr multiscale images). Only the chunks a read needs are fetched. Group stores expose members via `proxy["array_name"]`.
2. **`OmeZarrProxy` helpers** — `num_levels`, `get_level`, `thumbnail`, `iter_patches`, `random_patch`, and the new physical-coordinate `roi()` view.
3. **Loading** — the `zarrfolder` packaged builder (`load_dataset("zarrfolder", data_dir=...)`), the fast `load_zarr_dataset()` local loader, and **remote `data_dir`** (e.g. `hf://buckets/...`) with `streaming=True`.
4. **Performance** — transparent caching: open stores are reused, and decoded chunks are memoized in a bytes-bounded LRU (`DATASETS_ZARR_CHUNK_CACHE_SIZE`, default 256 MiB).
5. **Training** — `ZarrCollator` turns batches of lazy proxies into `pixel_values`/`labels` tensors (map-style `DataLoader` or streaming `batch()`), extracting patches concurrently (up to 8 workers).
6. **Sharing** — `push_to_hub_zarr()` uploads stores to a **Storage Bucket** (`hf://buckets/...`, resumable) or a regular dataset repo, plus the index-repo pattern.

> **Two modes.** Sections 0–5 run fully offline on synthetic stores created here. Section 6 needs internet + a Hub login and is clearly marked.

## 0. Setup

**Installation** (run once, from the branch root):

```bash
pip install -e .            # the PR branch
pip install "zarr>=3.0.0"   # required for the Zarr feature
pip install torch matplotlib  # optional: collator tensors, inline thumbnails
```

On **Windows**, torch's bundled `libomp.dll` can conflict with the copy bundled by `numcodecs.blosc` (zarr reads load it first). `ZarrCollator` sets `KMP_DUPLICATE_LIB_OK=TRUE` itself before importing torch, but we set it up-front here so plain `torch` imports also work.

In [1]:
import os
from pathlib import Path

if os.name == "nt":
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import zarr

import datasets

print("datasets:", datasets.__version__)
print("zarr:", zarr.__version__)
# -> datasets: 5.0.2.dev0   (this branch)
# -> zarr: 3.x.x

datasets: 5.0.2.dev0
zarr: 3.1.6


In [2]:
# Optional: tune the decoded-chunk cache (bytes). Must be set BEFORE `import datasets`.
# os.environ["DATASETS_ZARR_CHUNK_CACHE_SIZE"] = str(64 * 1024 * 1024)  # 64 MiB
# os.environ["DATASETS_ZARR_CHUNK_CACHE_SIZE"] = "0"                    # disable caching

DATA_ROOT = Path("zarr_demo_data")  # all synthetic stores live here
DATA_ROOT.mkdir(parents=True, exist_ok=True)

## 1. The `Zarr` feature — lazy proxies

The feature stores **only the path** in Arrow. Decoding returns a `ZarrProxy` that opens the store on first access (`.shape`, `.dtype`, `[...]`, ...) and keeps it cached afterwards. It auto-detects the store kind: plain array → `ZarrArrayProxy`, plain group → `ZarrGroupProxy`, OME-Zarr → `OmeZarrProxy` (section 2).

In [3]:
def create_plain_store(path, shape=(512, 512), dtype="float32", chunks=(128, 128), seed=0):
    """Write a small plain Zarr *group* with a `data` array and a `mask` array."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    data = np.random.default_rng(seed).normal(size=shape).astype(dtype)
    root = zarr.open_group(str(path), mode="w")
    arr = root.create_array("data", shape=shape, dtype=dtype, chunks=chunks)
    arr[:] = data
    mask = root.create_array("mask", shape=shape, dtype="uint8", chunks=chunks)
    mask[:] = (data > 0).astype("uint8")
    return path


plain_store = create_plain_store(DATA_ROOT / "plain" / "scan.zarr")
print(plain_store)
# -> zarr_demo_data\plain\scan.zarr

zarr_demo_data\plain\scan.zarr


In [4]:
from datasets import Dataset, Features, Zarr

ds = Dataset.from_dict({"scan": [str(plain_store)]}, features=Features({"scan": Zarr()}))

print(ds.features)
print(ds[0]["scan"])
# -> {'scan': Zarr(decode=True)}
# -> ZarrProxy(path='zarr_demo_data\\plain\\scan.zarr')   <- not opened yet!

{'scan': Zarr(decode=True)}
ZarrProxy(path='zarr_demo_data\\plain\\scan.zarr')


In [5]:
proxy = ds[0]["scan"]

proxy  # a group store resolves to ZarrGroupProxy
# -> ZarrGroupProxy(path='zarr_demo_data\\plain\\scan.zarr', members=['data', 'mask'])

# Groups have no shape of their own: navigate to an array member first.
data = proxy["data"]
data.shape, data.dtype, data.ndim, data.chunks
# -> ((512, 512), dtype('float32'), 2, (128, 128))

ValueError: ZarrGroup does not have a shape. Access individual arrays via group[key].shape

In [ ]:
# `data` is its own lazy ZarrArrayProxy (from the group navigation above).
print(data)
print(data.shape, data.dtype, data.chunks)

# Slicing loads ONLY the chunks that overlap the slice (2 chunks here).
patch = data[10:20, 30:40]
print(patch.shape, patch.dtype)
# -> ZarrArrayProxy(path='zarr_demo_data\\plain\\scan.zarr/data', shape=(512, 512), dtype=float32)
# -> (512, 512) float32 (128, 128)
# -> (10, 10) float32

In [ ]:
# Patch iteration: (coordinates, patch) pairs, strided -> 64 patches here.
n = 0
for coords, p in data.iter_patches((128, 128), stride=(64, 64)):
    if n < 2:
        print(coords, p.shape)
    n += 1
print("total patches:", n)

# Reproducible random patch via a seeded numpy Generator.
rng = np.random.default_rng(7)
print("random patch:", data.random_patch((128, 128), rng=rng).shape)
print("again, same rng:", data.random_patch((128, 128), rng=np.random.default_rng(7)).shape)
# -> (0, 0) (128, 128)
# -> (0, 64) (128, 128)
# -> total patches: 64
# -> random patch: (128, 128)
# -> again, same rng: (128, 128)

In [ ]:
# asarray() materializes the FULL array — only for small stores!
full = data.asarray()
print(full.shape, full.dtype)

# In a Jupyter notebook, displaying a small 2D+ array proxy renders an
# inline thumbnail preview via _repr_html_ (no full download):
# data
# -> (512, 512) float32

## 2. OME-Zarr multiscale helpers

An OME-Zarr (NGFF) group declares `multiscales` metadata with per-level `coordinateTransformations` (scale/translation, in physical units like µm) plus optional `omero` channel labels. Both NGFF v0.4 (metadata at the top of `.zattrs`) and v0.5 (nested under `"ome"`) are detected.

In [ ]:
def create_ome_store(path, levels=3, shape=(1, 256, 256), base_chunks=(1, 128, 128), seed=0):
    """Write a synthetic OME-Zarr v0.4 store: C-Y-X axes, 2x downsampling per level."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    scales = [(1.0, 0.5, 0.5), (2.0, 1.0, 1.0), (4.0, 2.0, 2.0)]
    root = zarr.open_group(str(path), mode="w")
    root.attrs["multiscales"] = [
        {
            "version": "0.4",
            "axes": [
                {"name": "c", "type": "channel"},
                {"name": "y", "type": "space", "unit": "micrometer"},
                {"name": "x", "type": "space", "unit": "micrometer"},
            ],
            "datasets": [
                {
                    "path": str(i),
                    "coordinateTransformations": [{"type": "scale", "scale": scales[i]}],
                }
                for i in range(levels)
            ],
        }
    ]
    root.attrs["omero"] = {"channels": [{"label": "DAPI"}, {"label": "GFP"}]}
    rng = np.random.default_rng(seed)
    for i, sc in enumerate(scales[:levels]):
        lvl_shape = tuple(int(s) for s in (shape[0], shape[1] // (2**i), shape[2] // (2**i)))
        chunks = tuple(min(c, s) for c, s in zip(base_chunks, lvl_shape))
        arr = root.create_array(str(i), shape=lvl_shape, dtype="uint16", chunks=chunks)
        arr[:] = rng.integers(0, 1000, size=lvl_shape, dtype="uint16")
    return path


ome_store = create_ome_store(DATA_ROOT / "ome" / "tissue.zarr")
print(ome_store)
# -> zarr_demo_data\ome\tissue.zarr

In [ ]:
ome = Dataset.from_dict({"scan": [str(ome_store)]}, features=Features({"scan": Zarr()}))[0]["scan"]

print(ome)  # auto-detected as OME-Zarr
print("num_levels:", ome.num_levels)
print("axes:", ome.axis_names, ome.axis_types)
print("level-0 scale (physical units per voxel):", ome.scale)
print("channel labels:", ome.channel_names)
# -> ZarrProxy(path='zarr_demo_data\\ome\\tissue.zarr')  <- repr before first access; opened lazily
# -> num_levels: 3
# -> axes: ['c', 'y', 'x'] ['channel', 'space', 'space']
# -> level-0 scale (physical units per voxel): [1.0, 0.5, 0.5]
# -> channel labels: ['DAPI', 'GFP']

In [ ]:
# Resolution levels: level 0 = highest, -1 = lowest (smallest).
for lvl in [0, 1, -1]:
    arr = ome.get_level(lvl)
    print(f"level {lvl}: shape={arr.shape} chunks={arr.chunks}")

thumb = ome.thumbnail(level=-1)  # loads the full (small) level into memory
print("thumbnail:", thumb.shape, thumb.dtype)
# -> level 0: shape=(1, 256, 256) chunks=(1, 128, 128)
# -> level 1: shape=(1, 128, 128) chunks=(1, 128, 128)
# -> level -1: shape=(1, 64, 64) chunks=(1, 64, 64)
# -> thumbnail: (1, 64, 64) uint16

In [ ]:
# iter_patches / random_patch work at any level (patch_size = spatial dims).
n = 0
for coords, p in ome.iter_patches((64, 64), stride=(64, 64), level=1):
    if n < 2:
        print(coords, p.shape)
    n += 1
print("total patches at level 1:", n)

print("random patch at level 0:", ome.random_patch((64, 64), level=0, rng=np.random.default_rng(3)).shape)
# -> (0, 0) (1, 64, 64)     <- leading channel axis is kept, patch applies to y/x
# -> (0, 64) (1, 64, 64)
# -> total patches at level 1: 4
# -> random patch at level 0: (1, 64, 64)

In [ ]:
# roi(): read a region given in PHYSICAL coordinates (µm here).
# Level-0 scale is [1.0, 0.5, 0.5] µm/voxel, so 1..3 µm  ==  pixels 2..6 on y/x.
region = ome.roi((0.0, 1.0, 1.0), (1.0, 3.0, 3.0), level=0)
print("roi 1-3 um @ level 0:", region.shape)

# The same physical region on a lower level picks up that level's own scale.
region_l1 = ome.roi((0.0, 1.0, 1.0), (1.0, 3.0, 3.0), level=1)  # scale [2, 1, 1]
print("roi 1-3 um @ level 1:", region_l1.shape)

# Open-ended (None) bounds and out-of-range regions are clipped to the array.
print("open start:", ome.roi((None, None, None), (1.0, 1.0, 1.0), level=0).shape)
print("clipped:", ome.roi((0.0, -10.0, -10.0), (1.0, 1000.0, 1000.0), level=0).shape)
# -> roi 1-3 um @ level 0: (1, 4, 4)
# -> roi 1-3 um @ level 1: (1, 2, 2)
# -> open start: (1, 2, 2)
# -> clipped: (1, 256, 256)

## 3. Loading: `zarrfolder` builder + `load_zarr_dataset`

Two ways to load a folder of stores:

- **`load_zarr_dataset(data_dir)`** — fast local discovery: walks directory **names** and stops at `.zarr` dirs, O(number of stores) instead of O(all chunk files). Labels are inferred from parent directories (`drop_labels=False`).
- **`load_dataset("zarrfolder", data_dir=...)`** — the standard builder; also supports **remote** `data_dir` (`hf://`, `s3://`, ...) and metadata files (`metadata.csv` / `.jsonl` / `.parquet`).

In [ ]:
label_root = DATA_ROOT / "labeled"
for i, label in enumerate(["healthy", "diseased"]):
    for j in range(2):
        create_ome_store(
            label_root / label / f"scan{j + 1}.zarr",
            seed=100 * i + j,
        )

print(sorted(str(p.relative_to(label_root)) for p in label_root.rglob("*.zarr")))
# -> ['diseased\\scan1.zarr', 'diseased\\scan2.zarr', 'healthy\\scan1.zarr', 'healthy\\scan2.zarr']

In [ ]:
from datasets.utils.zarr_utils import load_zarr_dataset

lds = load_zarr_dataset(str(label_root), drop_labels=False)
print(lds.column_names)
print(lds.features)
print(lds[0]["label"], lds[0]["zarr"].shape, lds[0]["zarr"].num_levels)
# -> ['zarr', 'label']
# -> {'zarr': Zarr(decode=True), 'label': ClassLabel(names=['diseased', 'healthy'])}
# -> 0 (1, 256, 256) 3   <- label 0 == 'diseased' (stores are sorted)

In [ ]:
# Metadata files add extra columns (drop_metadata=False to keep them).
# `file_name` is resolved relative to the folder that holds the stores.
meta_root = DATA_ROOT / "labeled_meta"
for i, label in enumerate(["healthy", "diseased"]):
    for j in range(2):
        create_ome_store(meta_root / f"scan{2 * i + j + 1}.zarr", seed=100 * i + j)

import csv

with open(meta_root / "metadata.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["file_name", "label", "patient_id", "age"])
    for i, label in enumerate(["healthy", "diseased"]):
        for j in range(2):
            n = 2 * i + j + 1
            w.writerow([f"scan{n}.zarr", label, f"P-{n}", 30 + n])

from datasets import load_dataset

md_ds = load_dataset("zarrfolder", data_dir=str(meta_root), drop_metadata=False)
print(md_ds.column_names)
print(md_ds["train"][0])
# -> {'train': ['zarr', 'label', 'patient_id', 'age']}
# -> {'zarr': ZarrProxy(path='zarr_demo_data/labeled_meta\\scan1.zarr'), 'label': 'healthy', 'patient_id': 'P-1', 'age': 31}

In [ ]:
# The same builder streams: rows yield lazy proxies, stores are opened on access.
stream_ds = load_dataset("zarrfolder", data_dir=str(label_root), streaming=True)
row = next(iter(stream_ds["train"]))
print(row["zarr"])
print(row["zarr"].shape, row["zarr"].num_levels)
# -> ZarrProxy(path='zarr_demo_data\\labeled\\diseased\\scan1.zarr')
# -> (1, 256, 256) 3

### 3b. Live: remote `data_dir` (needs internet, no auth)

A real OME-Zarr store (6 multiscale levels of a tissue image, ~7.8k chunk files) is publicly available in a Hub Storage Bucket. Streaming fetches only the chunks each read needs.

In [ ]:
remote_ds = load_dataset(
    "zarrfolder",
    data_dir="hf://buckets/tiagolubiana/ome-vip-dataset-bucket",
    streaming=True,
)
row = next(iter(remote_ds["train"]))
proxy = row["zarr"]
print(proxy.shape, proxy.num_levels)
print(proxy.roi((0, 0, 0), (1, 20, 20)).shape)  # 20 um x 20 um at level 0
# -> ((1937, 2048, 2048), 6)
# -> (1, 20, 20)  <- exact shape follows the level-0 axis scales (1 for the leading axis)

## 4. Performance: what the caches do

Two internal, read-only-safe caches make repeated/overlapping reads cheap (read-only stores can never go stale):

- **`STORE_REGISTRY`** (LRU, 32 entries): one open Zarr root per `(path, token, storage_options)` — no re-opening / re-reading metadata per decode.
- **`CHUNK_CACHE`** (bytes-bounded LRU, default 256 MiB): memoizes **decoded** chunks keyed by `(array_path, chunk_coords)`. The OS page cache can save file reads but never decode work; this cache makes overlapping patches/ROIs decode each shared chunk exactly once.

Size comes from the `DATASETS_ZARR_CHUNK_CACHE_SIZE` env var (bytes, `0` disables; set before `import datasets`).

Demo — two overlapping reads touch the same chunk; it is fetched and decoded once:

In [ ]:
from datasets.features.zarr_cache import CHUNK_CACHE, STORE_REGISTRY

CHUNK_CACHE.clear()  # reset for the demo
lvl0 = ome.get_level(0)  # chunks are (1, 128, 128)

a = lvl0[0, 0:100, 0:100]    # chunk (0, 0, 0)
b = lvl0[0, 0:100, 50:150]   # chunks (0, 0, 0) + (0, 0, 1)  -> (0, 0, 0) is a cache hit

print("cached chunks:", len(CHUNK_CACHE._cache))
print("cached bytes:", CHUNK_CACHE._bytes)
print("open stores:", len(STORE_REGISTRY._cache))
# -> cached chunks: 2        (chunk (0, 0, 0) decoded exactly once)
# -> cached bytes: 65536     (2 chunks x 1*128*128*2 bytes)
# -> open stores: >= 1       (one root per (path, token, storage_options))

## 5. Training: `ZarrCollator`

`ZarrCollator(patch_size=(...), level=0, label_column="label")` turns batches of lazy proxies into stacked tensors:

- keys `pixel_values` (and `labels` when the label column is present); torch tensors, numpy fallback without torch;
- extracts patches **concurrently** (up to 8 threads) with per-sample child RNGs for reproducibility;
- reads go through the chunk cache, so overlapping patches across the batch fetch each chunk once;
- works both as a `DataLoader` `collate_fn` (map-style) and on streaming `batch()` outputs.

In [ ]:
from torch.utils.data import DataLoader
from datasets.utils.zarr_utils import ZarrCollator

collator = ZarrCollator(
    patch_size=(64, 64),
    level=0,
    label_column="label",
    rng=np.random.default_rng(42),  # reproducible patch positions
)
loader = DataLoader(lds, batch_size=2, collate_fn=collator)

batch = next(iter(loader))
print(batch["pixel_values"].shape, batch["pixel_values"].dtype)
print(batch["labels"])
# -> torch.Size([2, 1, 64, 64]) torch.uint16
# -> tensor([0, 0])   <- first batch is the two 'diseased' rows

In [ ]:
# Streaming: IterableDataset.batch() yields *columnar* dicts; the collator
# normalizes them into per-sample dicts and extracts concurrently.
# stream_ds is the labeled folder, so labels are collated too.
for batch in stream_ds["train"].batch(batch_size=4):
    out = collator(batch)
    print(out["pixel_values"].shape)
    print(out["labels"])
    break
# -> torch.Size([4, 1, 64, 64])
# -> tensor([0, 0, 0, 0])

## 6. Sharing: `push_to_hub_zarr` (needs a Hub login / `HF_TOKEN`)

`push_to_hub_zarr(local_path, repo_id, ...)` routes on the `repo_id`:

- `repo_id="buckets/<namespace>/<bucket>"` → **Storage Bucket** (recommended for large stores): bucket created on demand, files synced with resumable `HfApi.sync_bucket`, no per-directory file limits. Returns an `hf://buckets/...` path.
- any other `repo_id` → regular dataset repo via `upload_folder`/`upload_large_folder`. Zarr stores are **never re-chunked**; if the store exceeds `file_limit` (default 10,000 files) a warning recommends a bucket.

```python
from huggingface_hub import login
login()  # or set HF_TOKEN env var
```

In [ ]:
from datasets.utils.zarr_utils import push_to_hub_zarr

# Large stores -> Storage Bucket (resumable sync, no file-count limits).
url = push_to_hub_zarr(
    str(ome_store),
    repo_id="buckets/<your-username>/zarr-demo",
)
print(url)
# -> hf://buckets/<bucket-id>/tissue.zarr   (store name = path_in_repo default)

# Small stores can go to a regular dataset repository.
url2 = push_to_hub_zarr(str(plain_store), repo_id="<your-username>/zarr-demo")
print(url2)
# -> https://huggingface.co/datasets/<your-username>/zarr-demo

In [ ]:
# Recommended pattern: keep raw data in a bucket/repo, and push a lightweight
# INDEX dataset (one row per sample, a `scan` path column) to the Hub.
from datasets import Dataset, DatasetDict

record = {
    "sample_id": "sample_001",
    "scan": "hf://buckets/<your-username>/zarr-demo/tissue.zarr",
}
index_ds = DatasetDict({"train": Dataset.from_list([record])})
index_ds.push_to_hub("<your-username>/zarr-index", private=True)

# Later, anyone (with access) streams it and casts the path column to Zarr:
# idx = load_dataset("<your-username>/zarr-index", streaming=True)
# idx = idx.cast_column("scan", Zarr())
# proxy = next(iter(idx["train"]))["scan"]
# proxy.shape  # -> (1, 256, 256)

## 7. Appendix — gotchas & pointers

- **Version**: requires `zarr>=3.0.0` (`Zarr` decode raises otherwise).
- **Chunk cache**: `DATASETS_ZARR_CHUNK_CACHE_SIZE` (bytes, default 256 MiB, `0` = off) must be set before `import datasets`.
- **Windows**: set `KMP_DUPLICATE_LIB_OK=TRUE` before importing torch (done in section 0; `ZarrCollator` also sets it defensively).
- **Never re-chunked**: `push_to_hub_zarr` uploads stores as-is; huge chunk-file counts belong in a Storage Bucket.
- **fsspec caching**: suffix byte-range reads (used by the sharding codec) are incompatible with `simplecache` wrappers — pass raw stores, not cached wrappers.
- **More docs**: `docs/source/zarr_dataset.mdx` (guides), `docs/source/stream.mdx` (streaming tip).

```python
# Optional cleanup of all synthetic data written by this notebook:
import shutil

shutil.rmtree(DATA_ROOT, ignore_errors=True)
```